0. Importing - Loading Data/Libraries

In [2]:
# @title
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_URL = (
    "https://raw.githubusercontent.com/fkottas/Teaching-Courses/main/"
    "Machine%20Learning%20in%20Finance/2026-2027/lectures/lecture-00-01/"
    "data/credit/default_of_credit_card_clients.csv.gz"
)

credit = pd.read_csv(DATA_URL, compression="gzip")
credit = credit.rename(columns={"default payment next month": "default_next_month"})
print("shape:", credit.shape)
print(credit.head(3).to_string(index=False))

shape: (30000, 25)
 ID  LIMIT_BAL  SEX  EDUCATION  MARRIAGE  AGE  PAY_0  PAY_2  PAY_3  PAY_4  PAY_5  PAY_6  BILL_AMT1  BILL_AMT2  BILL_AMT3  BILL_AMT4  BILL_AMT5  BILL_AMT6  PAY_AMT1  PAY_AMT2  PAY_AMT3  PAY_AMT4  PAY_AMT5  PAY_AMT6  default_next_month
  1      20000    2          2         1   24      2      2     -1     -1     -2     -2       3913       3102        689          0          0          0         0       689         0         0         0         0                   1
  2     120000    2          2         2   26     -1      2      0      0      0      2       2682       1725       2682       3272       3455       3261         0      1000      1000      1000         0      2000                   1
  3      90000    2          2         2   34      0      0      0      0      0      0      29239      14027      13559      14331      14948      15549      1518      1500      1000      1000      1000      5000                   0


1. Dataset Documentation & Metadata

In [3]:
# @title
import numpy as np
import pandas as pd

DATA_URL = (
    "https://raw.githubusercontent.com/fkottas/Teaching-Courses/main/"
    "Machine%20Learning%20in%20Finance/2026-2027/lectures/lecture-00-01/"
    "data/credit/default_of_credit_card_clients.csv.gz"
)

credit = pd.read_csv(DATA_URL, compression="gzip")

credit = credit.rename(columns={"default payment next month": "default_next_month"})

print(f"Loaded Shape: {credit.shape[0]:,} rows and {credit.shape[1]} columns")
credit.head(3)

Loaded Shape: 30,000 rows and 25 columns


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default_next_month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0


2. Comprehensive Data-Quality Audit

In [4]:
# @title
audit_summary = pd.DataFrame({
    'dtype': credit.dtypes.astype(str),
    'missing': credit.isna().sum(),
    'unique': credit.nunique(),
    'min': credit.min(numeric_only=True),
    'max': credit.max(numeric_only=True)
})
print(" COMPREHENSIVE DATA AUDIT TABLE ")
display(audit_summary)


print(f"\n1. Shape: {credit.shape[0]:,} rows and {credit.shape[1]} columns")
print(f"2. Total missing cells (isna): {credit.isna().sum().sum()}")
print(f"3. Exact row duplicates (entire row): {credit.duplicated().sum()}")
print(f"4. Unique ID count: {credit['ID'].nunique():,} (Matches total rows: {credit['ID'].nunique() == len(credit)})")

print("\n5. Target Balance ('default_next_month'):")
target_distribution = pd.DataFrame({
    'Count': credit['default_next_month'].value_counts(),
    'Percentage (%)': (credit['default_next_month'].value_counts(normalize=True) * 100).round(2)
})
target_distribution.index = ['No Default (0)', 'Default (1)']
display(target_distribution)

print("\n6. Coded Category Frequencies:")
print("\n--- EDUCATION ---")
print(credit['EDUCATION'].value_counts().sort_index())

print("\n--- MARRIAGE ---")
print(credit['MARRIAGE'].value_counts().sort_index())

print("\n--- SEX ---")
print(credit['SEX'].value_counts().sort_index())

 COMPREHENSIVE DATA AUDIT TABLE 


,dtype,missing,unique,min,max
ID,int64,0,30000,1,30000
LIMIT_BAL,int64,0,81,10000,1000000
SEX,int64,0,2,1,2
EDUCATION,int64,0,7,0,6
MARRIAGE,int64,0,4,0,3
AGE,int64,0,56,21,79
PAY_0,int64,0,11,-2,8
PAY_2,int64,0,11,-2,8
PAY_3,int64,0,11,-2,8
PAY_4,int64,0,11,-2,8



1. Shape: 30,000 rows and 25 columns
2. Total missing cells (isna): 0
3. Exact row duplicates (entire row): 0
4. Unique ID count: 30,000 (Matches total rows: True)

5. Target Balance ('default_next_month'):


,Count,Percentage (%)
No Default (0),23364,77.88
Default (1),6636,22.12



6. Coded Category Frequencies:

--- EDUCATION ---
EDUCATION
0       14
1    10585
2    14030
3     4917
4      123
5      280
6       51
Name: count, dtype: int64

--- MARRIAGE ---
MARRIAGE
0       54
1    13659
2    15964
3      323
Name: count, dtype: int64

--- SEX ---
SEX
1    11888
2    18112
Name: count, dtype: int64


3. Realistic Scenario: Incomplete Application and Strategic Information Omission (Adverse Selection). The missigness will target to solve the adverse selection problem.

In [5]:
# @title
demo_scenario = credit[['LIMIT_BAL', 'AGE', 'EDUCATION', 'MARRIAGE']].head(8).copy()

demo_scenario.loc[[1, 4], 'LIMIT_BAL'] = np.nan
demo_scenario.loc[[2, 5], 'AGE'] = np.nan
demo_scenario.loc[[3], 'EDUCATION'] = np.nan

print(" In-Memory Table with Controlled Missing Values ")
display(demo_scenario)

print("\nMissingness Rate per Feature:")
print(demo_scenario.isna().mean())

demo_scenario['LIMIT_BAL_was_missing'] = demo_scenario['LIMIT_BAL'].isna().astype(int)
demo_scenario['AGE_was_missing'] = demo_scenario['AGE'].isna().astype(int)

demo_scenario['LIMIT_BAL'] = demo_scenario['LIMIT_BAL'].fillna(demo_scenario['LIMIT_BAL'].median())
demo_scenario['AGE'] = demo_scenario['AGE'].fillna(demo_scenario['AGE'].median())

demo_scenario['EDUCATION'] = demo_scenario['EDUCATION'].fillna(demo_scenario['EDUCATION'].mode()[0])

print("\n=== Transformed Table (After Imputation and Indicators) ===")
display(demo_scenario[['LIMIT_BAL', 'LIMIT_BAL_was_missing', 'AGE', 'AGE_was_missing', 'EDUCATION', 'MARRIAGE']])

 In-Memory Table with Controlled Missing Values 


,LIMIT_BAL,AGE,EDUCATION,MARRIAGE
0,20000.0,24.0,2.0,1
1,NaN,26.0,2.0,2
2,90000.0,NaN,2.0,2
3,50000.0,37.0,NaN,1
4,NaN,57.0,2.0,1
5,50000.0,NaN,1.0,2
6,500000.0,29.0,1.0,2
7,100000.0,23.0,2.0,2



Missingness Rate per Feature:
LIMIT_BAL    0.250
AGE          0.250
EDUCATION    0.125
MARRIAGE     0.000
dtype: float64

=== Transformed Table (After Imputation and Indicators) ===


,LIMIT_BAL,LIMIT_BAL_was_missing,AGE,AGE_was_missing,EDUCATION,MARRIAGE
0,20000.0,0,24.0,0,2.0,1
1,70000.0,1,26.0,0,2.0,2
2,90000.0,0,27.5,1,2.0,2
3,50000.0,0,37.0,0,2.0,1
4,70000.0,1,57.0,0,2.0,1
5,50000.0,0,27.5,1,1.0,2
6,500000.0,0,29.0,0,1.0,2
7,100000.0,0,23.0,0,2.0,2


4. In credit risk datasets, checking only for exact full-row duplicates or repeated database keys (`ID`) is insufficient. Below, we first construct an in-memory controlled duplicate example, and then audit the full dataset to compare exact row duplicates against latent feature-level duplicates.

In [6]:
# @title
small_sample = credit.head(6).copy()
demo_dubs = pd.concat([small_sample, small_sample.iloc[[0]]], ignore_index=True)

print(" 1. Controlled In-Memory Duplicate Example ")
print(f"Sample shape after repeating first row: {demo_dubs.shape}")
print(f"Exact duplicate rows detected: {demo_dubs.duplicated().sum()}")
print(f"Duplicate ID count detected:   {demo_dubs['ID'].duplicated().sum()}")

predictor_cols = [c for c in credit.columns if c not in ['ID', 'default_next_month']]

exact_row_dubs = credit.duplicated().sum()
id_dubs = credit['ID'].duplicated().sum()
latent_feature_dups = credit.duplicated(subset=predictor_cols, keep=False).sum()

print("\n 2. Full Dataset Duplicate Audit ")
print(f"Exact full-row duplicates (including ID): {exact_row_dubs}")
print(f"Repeated IDs:                             {id_dubs}")
print(f"Latent duplicates (identical X features, different ID): {latent_feature_dups}")

 1. Controlled In-Memory Duplicate Example 
Sample shape after repeating first row: (7, 25)
Exact duplicate rows detected: 1
Duplicate ID count detected:   1

 2. Full Dataset Duplicate Audit 
Exact full-row duplicates (including ID): 0
Repeated IDs:                             0
Latent duplicates (identical X features, different ID): 108


5. Documented outlier flag on a numeric variable (AGE)

In [7]:
# @title
outlier_age = credit[['AGE']].copy()

q1_age = outlier_age['AGE'].quantile(0.25)
q3_age = outlier_age['AGE'].quantile(0.75)
iqr_age = q3_age - q1_age

lower_bound_age = q1_age - 1.5 * iqr_age
upper_bound_age = q3_age + 1.5 * iqr_age
p99_age = outlier_age['AGE'].quantile(0.99)

outlier_age['AGE_iqr_outlier'] = (
    (outlier_age['AGE'] < lower_bound_age) | (outlier_age['AGE'] > upper_bound_age)
).astype(int)

outlier_age['AGE_top1pct_flag'] = (outlier_age['AGE'] > p99_age).astype(int)

print("Outlier Diagnostics for AGE ")
print(f"Q1 (25th percentile): {q1_age:.1f} years")
print(f"Q3 (75th percentile): {q3_age:.1f} years")
print(f"IQR (Q3 - Q1):        {iqr_age:.1f} years")
print(f"Lower IQR Fence:      {lower_bound_age:.1f} years")
print(f"Upper IQR Fence:      {upper_bound_age:.1f} years")
print(f"99th Percentile:      {p99_age:.1f} years")
print(f"Minimum Observed:     {outlier_age['AGE'].min()} years")
print(f"Maximum Observed:     {outlier_age['AGE'].max()} years")

print(f"\nObservations flagged outside IQR Fences: {outlier_age['AGE_iqr_outlier'].sum()} ({outlier_age['AGE_iqr_outlier'].mean()*100:.2f}%)")
print(f"Observations flagged in Top 1% (> P99):  {outlier_age['AGE_top1pct_flag'].sum()} ({outlier_age['AGE_top1pct_flag'].mean()*100:.2f}%)")

print("\n Default Rate by AGE Outlier Flag (Economic Signal Check) ")
display(outlier_age.groupby('AGE_iqr_outlier').agg(['count', 'mean']))

Outlier Diagnostics for AGE 
Q1 (25th percentile): 28.0 years
Q3 (75th percentile): 41.0 years
IQR (Q3 - Q1):        13.0 years
Lower IQR Fence:      8.5 years
Upper IQR Fence:      60.5 years
99th Percentile:      60.0 years
Minimum Observed:     21 years
Maximum Observed:     79 years

Observations flagged outside IQR Fences: 272 (0.91%)
Observations flagged in Top 1% (> P99):  272 (0.91%)

 Default Rate by AGE Outlier Flag (Economic Signal Check) 


AGE            AGE_top1pct_flag     
                 count       mean            count mean
AGE_iqr_outlier                                        
0                29728  35.220264            29728  0.0
1                  272  64.474265              272  1.0

6. Separate numeric, categorical, indentifier and target columns

In [8]:
# @title
id_col = ['ID']
target_col = ['default_next_month']

categorical_cols = [
    'SEX', 'EDUCATION', 'MARRIAGE',
    'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6'
]

numeric_cols = [
    c for c in credit.columns
    if c not in id_col + target_col + categorical_cols
]

X = credit[categorical_cols + numeric_cols].copy()
y = credit[target_col[0]].copy()

total_assigned = len(id_col) + len(target_col) + len(categorical_cols) + len(numeric_cols)

print(" Column Role Separation Summary ")
print(f"Total dataset columns:     {credit.shape[1]}")
print(f"Total assigned columns:    {total_assigned}")
print("-" * 55)
print(f"1. Identifier ({len(id_col)}):        {id_col}")
print(f"2. Target ({len(target_col)}):            {target_col}")
print(f"3. Categorical ({len(categorical_cols)}):       {categorical_cols}")
print(f"4. Numeric ({len(numeric_cols)}):          {numeric_cols}")
print("-" * 55)
print(f"Feature matrix X shape:    {X.shape}")
print(f"Target vector y shape:     {y.shape}")

 Column Role Separation Summary 
Total dataset columns:     25
Total assigned columns:    25
-------------------------------------------------------
1. Identifier (1):        ['ID']
2. Target (1):            ['default_next_month']
3. Categorical (9):       ['SEX', 'EDUCATION', 'MARRIAGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
4. Numeric (14):          ['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']
-------------------------------------------------------
Feature matrix X shape:    (30000, 23)
Target vector y shape:     (30000,)


7.Spliting the data with a fixed random seed and target stratification.

In [9]:
# @title
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

split_summary = pd.DataFrame({
    'Subset': ['Full Dataset (100%)', 'Training Set (70%)', 'Test Set (30%)'],
    'Observations': [len(y), len(y_train), len(y_test)],
    'Default_Count': [y.sum(), y_train.sum(), y_test.sum()],
    'Default_Rate_%': [
        round(y.mean() * 100, 2),
        round(y_train.mean() * 100, 2),
        round(y_test.mean() * 100, 2)
    ]
})

print("Stratified 70/30 Train/Test Split Verification:")
print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}  | y_test shape:  {y_test.shape}\n")
display(split_summary)

Stratified 70/30 Train/Test Split Verification:
X_train shape: (21000, 23) | y_train shape: (21000,)
X_test shape:  (9000, 23)  | y_test shape:  (9000,)



,Subset,Observations,Default_Count,Default_Rate_%
0,Full Dataset (100%),30000,6636,22.12
1,Training Set (70%),21000,4645,22.12
2,Test Set (30%),9000,1991,22.12


8.Leakage-Safe Preprocessing Pipeline with ColumnTransformer

In [10]:
# @title
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_pip = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pip = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pip, numeric_cols),
        ('cat', categorical_pip, categorical_cols)
    ],
    remainder='drop'
)

X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

encoded_feature_names = preprocessor.get_feature_names_out()

X_train_processed_df = pd.DataFrame(
    X_train_transformed,
    columns=encoded_feature_names,
    index=X_train.index
)
X_test_processed_df = pd.DataFrame(
    X_test_transformed,
    columns=encoded_feature_names,
    index=X_test.index
)

print("ColumnTransformer Pipeline Execution Summary:")
print(f"Original X_train shape:    {X_train.shape}")
print(f"Transformed X_train shape: {X_train_processed_df.shape}")
print(f"Original X_test shape:     {X_test.shape}")
print(f"Transformed X_test shape:  {X_test_processed_df.shape}\n")

print("Preview of Transformed Training Features (First 3 Rows):")
display(X_train_processed_df.head(3).round(3))

ColumnTransformer Pipeline Execution Summary:
Original X_train shape:    (21000, 23)
Transformed X_train shape: (21000, 91)
Original X_test shape:     (9000, 23)
Transformed X_test shape:  (9000, 91)

Preview of Transformed Training Features (First 3 Rows):


,num__LIMIT_BAL,num__AGE,num__BILL_AMT1,num__BILL_AMT2,num__BILL_AMT3,num__BILL_AMT4,num__BILL_AMT5,num__BILL_AMT6,num__PAY_AMT1,num__PAY_AMT2,...,cat__PAY_6_-2,cat__PAY_6_-1,cat__PAY_6_0,cat__PAY_6_2,cat__PAY_6_3,cat__PAY_6_4,cat__PAY_6_5,cat__PAY_6_6,cat__PAY_6_7,cat__PAY_6_8
11018,-0.058,-0.373,1.425,1.505,1.578,1.706,1.908,1.964,0.021,0.004,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1710,-0.520,-0.591,-0.310,-0.228,-0.068,0.128,0.168,0.231,-0.040,0.197,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4618,-0.674,-0.700,0.306,0.368,0.252,0.033,0.031,0.067,-0.143,-0.165,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


9. Displaying transformed shapes and feature names.

Explanation: How Our Workflow Prevents Data Leakage

Data leakage happens when information from the test set accidentally sneaks into the training process, giving the model an unfair advantage—like a student seeing the final exam questions while studying. Our workflow completely prevents this by following a strict order of operations. First, we dropped the non-predictive `ID` column and separated the target variable (`y`) from the features (`X`) so the answer key never mixes with the inputs. Second, we split the dataset into 70% training (21,000 rows) and 30% testing (9,000 rows) before calculating any averages, medians, or standard deviations, immediately locking the test set away. Third, we ran `.fit_transform()` exclusively on `X_train` so the pipeline learned all statistical parameters only from the training borrowers, and then applied `.transform()` on `X_test` using those exact training rules without recalculating anything. Finally, by bundling all imputation, scaling, and encoding steps inside a single `ColumnTransformer` with `handle_unknown="ignore"`, we avoided manually altering the full dataset in advance and ensured that even unseen categories in the test set are handled safely without peeking at the test data.

In [12]:
# @title
feature_names = preprocessor.get_feature_names_out()

num_out_names = [name for name in feature_names if name.startswith('num__')]
cat_out_names = [name for name in feature_names if name.startswith('cat__')]

print("Matrix Shapes Before and After Preprocessing:")
print(f"Raw X_train shape:         {X_train.shape}")
print(f"Transformed X_train shape: {X_train_transformed.shape}")
print(f"Raw X_test shape:          {X_test.shape}")
print(f"Transformed X_test shape:  {X_test_transformed.shape}\n")

print("Generated Feature Names Breakdown:")
print(f"Total output features:     {len(feature_names)}")
print(f"Scaled numeric features:   {len(num_out_names)}")
print(f"One-hot dummy features:    {len(cat_out_names)}\n")

print("List of Scaled Numeric Feature Names (num__):")
print(num_out_names)

print("\nFirst 15 One-Hot Encoded Categorical Feature Names (cat__):")
print(cat_out_names[:15])

print("\nComplete List of All Transformed Feature Names:")
print(list(feature_names))

Matrix Shapes Before and After Preprocessing:
Raw X_train shape:         (21000, 23)
Transformed X_train shape: (21000, 91)
Raw X_test shape:          (9000, 23)
Transformed X_test shape:  (9000, 91)

Generated Feature Names Breakdown:
Total output features:     91
Scaled numeric features:   14
One-hot dummy features:    77

List of Scaled Numeric Feature Names (num__):
['num__LIMIT_BAL', 'num__AGE', 'num__BILL_AMT1', 'num__BILL_AMT2', 'num__BILL_AMT3', 'num__BILL_AMT4', 'num__BILL_AMT5', 'num__BILL_AMT6', 'num__PAY_AMT1', 'num__PAY_AMT2', 'num__PAY_AMT3', 'num__PAY_AMT4', 'num__PAY_AMT5', 'num__PAY_AMT6']

First 15 One-Hot Encoded Categorical Feature Names (cat__):
['cat__SEX_1', 'cat__SEX_2', 'cat__EDUCATION_0', 'cat__EDUCATION_1', 'cat__EDUCATION_2', 'cat__EDUCATION_3', 'cat__EDUCATION_4', 'cat__EDUCATION_5', 'cat__EDUCATION_6', 'cat__MARRIAGE_0', 'cat__MARRIAGE_1', 'cat__MARRIAGE_2', 'cat__MARRIAGE_3', 'cat__PAY_0_-2', 'cat__PAY_0_-1']

Complete List of All Transformed Feature Name